# Kaggle Dataset 精简版 YOLO 训练

本 Notebook 将直接从通过 **Kaggle Add Data** 挂载的数据集进行训练，移除了手动上传、断点续训、Roboflow和云盘等功能，保证代码精简明了。

## 使用前提
1. 将包含 `images`, `labels`, `data.yaml` 及 `training_config.json` 的压缩包在 Kaggle 的 Datasets 页面创建一个新数据集。
2. 在右侧面板点击 **`+ Add Input`** 将刚创建的数据集挂载到当前 Notebook。
3. 在右侧面板开启 GPU（Accelerator 选 GPU T4 x2 或 P100）。

## 1) 环境准备

In [ ]:
# 1. 检查 GPU
!nvidia-smi

# 2. 安装必要依赖
!pip install -q -U ultralytics pyyaml

# 3. 公共路径准备
import os, glob, shutil, json
import yaml
from ultralytics import YOLO

WORK_ROOT = '/kaggle/working/InsightYOLO'
RUNS_PROJECT = os.path.join(WORK_ROOT, 'runs')
EXPORTS_ROOT = os.path.join(WORK_ROOT, 'exports')

os.makedirs(RUNS_PROJECT, exist_ok=True)
os.makedirs(EXPORTS_ROOT, exist_ok=True)

print(f'✅ RUNS_PROJECT: {RUNS_PROJECT}')
print(f'✅ EXPORTS_ROOT: {EXPORTS_ROOT}')

## 2) 寻找挂载的数据集并解析配置

In [ ]:
# 自动获取 Kaggle Input 中的 data.yaml
# 挂载后的数据集通常在 /kaggle/input 下
yaml_candidates = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
if not yaml_candidates:
    raise FileNotFoundError('在 /kaggle/input/ 未找到 data.yaml，请确认是否已在右侧 Add Input 挂载数据集！')

# 取路径最短的，防止深层无用文件干扰
original_data_yaml = sorted(yaml_candidates, key=len)[0]
dataset_path = os.path.dirname(original_data_yaml)
print(f'✅ 找到数据集目录: {dataset_path}')

# 读取并修正 data.yaml 为绝对路径（存至 /kaggle/working/，因为 /kaggle/input 是只读的）
with open(original_data_yaml, 'r', encoding='utf-8') as f:
    yaml_cfg = yaml.safe_load(f) or {}

base = yaml_cfg.get('path')
if not base or str(base).strip() in ('.', './'):
    base_dir = dataset_path
else:
    base = str(base).strip()
    base_dir = base if os.path.isabs(base) else os.path.normpath(os.path.join(dataset_path, base))

for key in ('train', 'val', 'test'):
    if key in yaml_cfg and yaml_cfg.get(key):
        p = str(yaml_cfg[key]).strip()
        if not os.path.isabs(p):
            if p.startswith('./'):
                p = p[2:]
            p = os.path.normpath(os.path.join(base_dir, p))
        yaml_cfg[key] = p.replace('\\', '/')

yaml_cfg.pop('path', None)

resolved_yaml = os.path.join(WORK_ROOT, 'data.resolved.yaml')
with open(resolved_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(yaml_cfg, f, allow_unicode=True, sort_keys=False)

print(f'✅ 生成修正后的 data.yaml: {resolved_yaml}')

# 读取 Insight 导出的 training_config.json
train_cfg = {
    'yolo_version': 'yolov8',
    'model_size': 'n',
    'epochs': 100,
    'batch_size': 16,
    'img_size': 640,
    'patience': 50,
    'workers': 8,
    'device': 0
}

cfg_path = os.path.join(dataset_path, 'training_config.json')
if os.path.exists(cfg_path):
    with open(cfg_path, 'r', encoding='utf-8') as f:
        file_cfg = json.load(f)
    train_cfg.update(file_cfg)
    print(f'✅ 已读取训练参数: {cfg_path}')
else:
    print('⚠️ 未找到 training_config.json，将使用默认参数')

model_family = str(train_cfg.get('yolo_version', 'yolov8'))
WEIGHT_MAP = {'yolov11': 'yolo11', 'yolo11': 'yolo11', 'yolo26': 'yolo26'}
weight_prefix = WEIGHT_MAP.get(model_family, model_family)
model_size = str(train_cfg.get('model_size', 'n')).lower()
model_name = f'{weight_prefix}{model_size}.pt'
run_name = f'{weight_prefix}{model_size}_train'

print(f'模型结构: {model_name}')
print('训练参数:', {k: train_cfg.get(k) for k in ['epochs','batch_size','img_size','patience','workers','device']})

## 3) 开始训练

In [ ]:
# GPU 设备处理（适配多卡）
raw_device = train_cfg.get('device', 0)
device_str = str(raw_device).replace('，', ',').replace(' ', '')
try:
    import torch
    gpu_count = torch.cuda.device_count()
    if gpu_count > 0 and ',' in device_str:
        req_gpus = [int(x) for x in device_str.split(',') if x]
        valid_gpus = [x for x in req_gpus if 0 <= x < gpu_count]
        device_str = ','.join(map(str, valid_gpus)) if valid_gpus else 0
except Exception:
    pass

print('🚀 开始新训练...')
model = YOLO(model_name)
results = model.train(
    data=resolved_yaml,
    epochs=int(train_cfg.get('epochs', 100)),
    imgsz=int(train_cfg.get('img_size', 640)),
    batch=int(train_cfg.get('batch_size', 16)),
    name=run_name,
    project=RUNS_PROJECT,
    exist_ok=True,
    save=True,
    patience=int(train_cfg.get('patience', 50)),
    workers=int(train_cfg.get('workers', 8)),
    device=device_str
)

print('🎉 训练完成！')

## 4) 导出 ONNX 及展示结果

In [ ]:
from IPython.display import Image, display

# DDP 模式下 results 可能为 None，手动构建路径
if results is not None and hasattr(results, 'save_dir'):
    save_dir = str(results.save_dir)
else:
    save_dir = os.path.join(RUNS_PROJECT, run_name)
print(f'结果目录: {save_dir}')

results_img = os.path.join(save_dir, 'results.png')
if os.path.exists(results_img):
    print('📈 训练曲线：')
    display(Image(results_img, width=900))

best_pt = os.path.join(save_dir, 'weights', 'best.pt')
if not os.path.exists(best_pt):
    print(f'⚠️ best.pt 不存在，导出 ONNX 失败: {best_pt}')
else:
    print('\n正在导出 ONNX...')
    best_model = YOLO(best_pt)
    imgsz = int(train_cfg.get('img_size', 640))
    export_path = best_model.export(
        format='onnx',
        imgsz=imgsz,
        simplify=True,
        opset=12,
        dynamic=False
    )
    
    output_name = f'detector_{weight_prefix}{model_size}.onnx'
    # 拷贝到 Kaggle 根工作目录，方便直接从右侧 Files 栏下载
    kaggle_download_path = os.path.join('/kaggle/working', output_name)
    shutil.copy(export_path, kaggle_download_path)
    
    file_size = os.path.getsize(kaggle_download_path) / 1024 / 1024
    print('\n✅ ONNX 导出完成！')
    print(f'文件已就绪，请直接在右侧 Files 面板下载: {kaggle_download_path}')
    print(f'文件大小: {file_size:.2f} MB')
    print(f'输入尺寸: {imgsz} x {imgsz}')

## 5) 简单推理验证评估

In [ ]:
import random

val_imgs = glob.glob(os.path.join(save_dir, 'val_batch*.jpg'))
if val_imgs:
    print('\n🖼️ 验证集标注重叠示例：')
    display(Image(val_imgs[0], width=900))

candidate_dirs = [
    os.path.join(dataset_path, 'test', 'images'),
    os.path.join(dataset_path, 'valid', 'images'),
    os.path.join(dataset_path, 'images', 'val'),
]

img_dir = next((d for d in candidate_dirs if os.path.exists(d)), None)
if img_dir:
    all_imgs = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png'))
    if all_imgs:
        sample_img = random.choice(all_imgs)
        print(f'\n🕵️‍♂️ 进行简单预测验证，测试图像：{os.path.basename(sample_img)}')
        
        pred_root = os.path.join(WORK_ROOT, 'predictions')
        infer_model = YOLO(best_pt)
        pred_results = infer_model.predict(
            source=sample_img,
            conf=0.25,
            save=True,
            project=pred_root,
            name='test',
            exist_ok=True
        )
        
        pred_img = glob.glob(os.path.join(pred_root, 'test', '*.jpg')) + glob.glob(os.path.join(pred_root, 'test', '*.png'))
        if pred_img:
            display(Image(pred_img[-1], width=700))
